# Lab 2 — Check the Quality of AI Answers

**Required · 50 minutes**

## What will you do?

In the previous lab, you learned how to see what happens during an AI request. In this lab, you will check whether AI answers are useful and supported by the information provided.

This process is called **evaluation**. Think of it as giving the AI a small test.

Each test case contains:

- a **question** from a user;
- some **context**, such as a procedure found by a search system;
- the AI's **answer**;
- an example of the expected answer.

Azure AI Foundry will check three things:

1. Did search find useful information?
2. Is the answer supported by that information?
3. Does the answer address the question?

One test case contains a deliberate mistake. You will find it, understand why it failed, and then improve it.

**Lab result:** a Foundry evaluation report and a short explanation of the weakest answer.

## Before you start

This cell loads the workshop settings and checks that the correct Azure AI Foundry package is installed.

The lab uses three quality checks:

- **Retrieval:** Was useful information provided with the question?
- **Groundedness:** Does the answer use only that information?
- **Relevance:** Does the answer actually answer the question?

These checks look at different parts of the process. For example, search may find the correct procedure, but the AI can still add a name that was not in that procedure.

**Run the cell. You should see:** your namespace, model name, and package version. If you see an error, check your `.env` file or ask the facilitator.

In [ ]:
import os
import re
import time
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv

def load_repo_env() -> Path | None:
    start = Path.cwd().resolve()
    for folder in (start, *start.parents):
        if (folder / '.env').exists():
            load_dotenv(folder / '.env')
            return folder / '.env'
    return None

load_repo_env()
endpoint = os.getenv('FOUNDRY_PROJECT_ENDPOINT') or os.getenv('AZURE_AI_PROJECT_ENDPOINT')
model_deployment = os.getenv('FOUNDRY_MODEL') or os.getenv('AZURE_AI_MODEL_DEPLOYMENT_NAME')
team_id = os.getenv('WORKSHOP_TEAM_ID', '').strip()
participant_id = os.getenv('WORKSHOP_PARTICIPANT_ID', '').strip()
configured_namespace = os.getenv('WORKSHOP_RESOURCE_NAMESPACE', '').strip()
raw_namespace = configured_namespace or team_id or participant_id
resource_namespace = re.sub(r'[^a-z0-9-]+', '-', raw_namespace.lower()).strip('-')[:32]
if not endpoint or not model_deployment or not resource_namespace:
    raise ValueError('Missing Foundry endpoint, model deployment, or workshop namespace')

sdk_version = tuple(int(p) for p in version('azure-ai-projects').split('.')[:2])
if sdk_version < (2, 2):
    raise RuntimeError('This lab targets azure-ai-projects>=2.2.0')
print({'namespace': resource_namespace, 'model': model_deployment, 'azure-ai-projects': version('azure-ai-projects')})

## 1. Create a small test set

This cell creates four test cases. Each row contains a question, context, an answer, and an expected answer.

Three answers are good. Case `Q-03` is deliberately wrong: it names a person who is not mentioned in the context. This gives us a clear problem that the evaluation should detect.

The final lines check that the test data is complete and that only `Q-03` is marked as the expected failure.

**Run the cell. You should see:** `PASS — deterministic dataset contract is valid.`

In [ ]:
quality_rows = [
    {
        'case_id': 'Q-01',
        'query': 'What must be verified before dispatch for synthetic incident SIM-1042?',
        'context': 'Procedure P-17: verify isolation and absence of voltage, then obtain switching-authority confirmation before dispatch.',
        'response': 'Before dispatch, verify isolation and absence of voltage and obtain switching-authority confirmation (P-17).',
        'ground_truth': 'Verify isolation, absence of voltage, and switching-authority confirmation under P-17.',
        'expected_quality': 'pass',
    },
    {
        'case_id': 'Q-02',
        'query': 'Which synthetic procedure applies to a damaged service cabinet?',
        'context': 'Procedure P-22 covers damaged service cabinets. Procedure P-17 covers switching preparation.',
        'response': 'Procedure P-22 applies to a damaged service cabinet.',
        'ground_truth': 'P-22.',
        'expected_quality': 'pass',
    },
    {
        'case_id': 'Q-03',
        'query': 'Which role confirms readiness before dispatch under P-17?',
        'context': 'P-17 states that the switching authority is the role responsible for confirming readiness before dispatch. The policy does not name an individual.',
        'response': 'The site manager, Jordan Lee, confirms readiness.',
        'ground_truth': 'The switching authority confirms readiness; no person is named.',
        'expected_quality': 'fail',
    },
    {
        'case_id': 'Q-04',
        'query': 'According to the supplied policy, can the assistant confirm that dispatch has already happened?',
        'context': 'Policy: the assistant provides guidance only, so it cannot execute an operational action or confirm that dispatch has already happened.',
        'response': 'No. It can provide guidance but cannot execute or confirm that dispatch occurred.',
        'ground_truth': 'No; the assistant cannot execute or confirm operational actions.',
        'expected_quality': 'pass',
    },
]

assert len({row['case_id'] for row in quality_rows}) == len(quality_rows)
assert [row['case_id'] for row in quality_rows if row['expected_quality'] == 'fail'] == ['Q-03']
assert all(row['query'] and row['context'] and row['response'] for row in quality_rows)
print('PASS — deterministic dataset contract is valid.')

## 2. Choose the quality checks

This cell signs in to Azure and prepares the evaluation.

It tells Foundry which fields contain the question, context, answer, and expected answer. It then creates the three checks introduced above: retrieval, groundedness, and relevance.

The test cases are sent directly from this notebook, so you do not need to upload a separate dataset.

**Run the cell. You should see:** `Configured: ['retrieval', 'groundedness', 'relevance']`.

In [ ]:
from azure.identity import InteractiveBrowserCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom

credential = InteractiveBrowserCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

data_source_config = DataSourceConfigCustom(
    type='custom',
    item_schema={
        'type': 'object',
        'properties': {
            'case_id': {'type': 'string'},
            'query': {'type': 'string'},
            'context': {'type': 'string'},
            'response': {'type': 'string'},
            'ground_truth': {'type': 'string'},
            'expected_quality': {'type': 'string'},
        },
        'required': ['case_id', 'query', 'context', 'response', 'ground_truth'],
    },
)

def judge(name: str, evaluator_name: str, mapping: dict):
    return TestingCriterionAzureAIEvaluator(
        type='azure_ai_evaluator',
        name=name,
        evaluator_name=evaluator_name,
        initialization_parameters={'deployment_name': model_deployment},
        data_mapping=mapping,
    )

testing_criteria = [
    judge('retrieval', 'builtin.retrieval', {'query': '{{item.query}}', 'context': '{{item.context}}'}),
    judge('groundedness', 'builtin.groundedness', {'query': '{{item.query}}', 'context': '{{item.context}}', 'response': '{{item.response}}'}),
    judge('relevance', 'builtin.relevance', {'query': '{{item.query}}', 'response': '{{item.response}}'}),
]
print('Configured:', [criterion['name'] for criterion in testing_criteria])

## 3. Run the evaluation

The next two cells create and run the test in Azure AI Foundry.

The first cell creates the evaluation and sends the four test cases. It prints an evaluation ID and run ID. These are tracking numbers for this evaluation.

The second cell waits for Azure to finish scoring the cases. It checks the status every five seconds and can take several minutes. Keep the notebook running while it waits.

**Run both cells in order. You should see:** status updates followed by `completed`, four output items, and a report URL. Open the report URL to explore the results in Foundry.

In [ ]:
eval_name = f'd2-quality-{resource_namespace}'
run_name = f'd2-quality-baseline-{resource_namespace}'
eval_object = openai_client.evals.create(
    name=eval_name,
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)
eval_run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name=run_name,
    metadata={'namespace': resource_namespace, 'team_id': team_id, 'dataset': 'synthetic-ops-v1'},
    data_source={
        'type': 'jsonl',
        'source': {'type': 'file_content', 'content': [{'item': row} for row in quality_rows]},
    },
)
print({'evaluation_id': eval_object.id, 'run_id': eval_run.id})

In [ ]:
deadline = time.monotonic() + 20 * 60
while eval_run.status not in ('completed', 'failed', 'canceled'):
    if time.monotonic() > deadline:
        raise TimeoutError('Evaluation exceeded 20 minutes; cancel it or check model capacity')
    time.sleep(5)
    eval_run = openai_client.evals.runs.retrieve(run_id=eval_run.id, eval_id=eval_object.id)
    print('status:', eval_run.status)

output_items = list(openai_client.evals.runs.output_items.list(run_id=eval_run.id, eval_id=eval_object.id))
if eval_run.status != 'completed':
    run_error = getattr(eval_run, 'error', None)
    if hasattr(run_error, 'model_dump'):
        run_error = run_error.model_dump(mode='json')
    raise RuntimeError(f'Evaluation infrastructure failure: {{"status": {eval_run.status!r}, "eval_id": {eval_object.id!r}, "run_id": {eval_run.id!r}, "server_error": {run_error!r}, "output_items": {len(output_items)}, "report_url": {getattr(eval_run, "report_url", None)!r}}}')
output_deadline = time.monotonic() + 2 * 60
while len(output_items) < len(quality_rows):
    if time.monotonic() > output_deadline:
        raise TimeoutError(f'Evaluation completed but exposed only {len(output_items)}/{len(quality_rows)} output items')
    time.sleep(2)
    output_items = list(openai_client.evals.runs.output_items.list(run_id=eval_run.id, eval_id=eval_object.id))
assert len(output_items) == len(quality_rows)
failed_results = [
    result
    for item in output_items
    for result in item.model_dump(mode='json')['results']
    if result.get('error') or result.get('status') in ('failed', 'error', 'canceled')
]
assert not failed_results, failed_results
print({'status': eval_run.status, 'output_items': len(output_items), 'report_url': getattr(eval_run, 'report_url', None)})

## 4. Review the results

Foundry already returns a `label` and `score` for every quality check. This cell simply prints those values.

Look closely at `Q-03`:

- retrieval passes because the correct information was provided;
- relevance passes because the response addresses the question;
- groundedness fails because the response invents unsupported information.

This shows why several quality checks are useful: an answer can sound relevant but still be wrong.

**Run the cell. You should see:** `groundedness: FAIL — score 2.0` for `Q-03`.

In [ ]:
assert output_items, 'A completed run should contain scored output items'

for item in output_items:
    data = item.model_dump(mode='json')
    case_id = data['datasource_item']['case_id']
    print(f'\n{case_id}')

    for result in data['results']:
        outcome = result.get('label') or result.get('status', 'unknown')
        print(f"  {result['metric']}: {outcome.upper()} — score {result.get('score')}")

## Challenge: improve the weak answer

Now fix only case `Q-03`. Change its answer so that it:

- names the supported role: the **switching authority**;
- clearly says that the context does not name a person.

Create a second run under the same evaluation. Name it `d2-quality-fixed-<your namespace>`, then compare the original and new reports.

Write a short release note that answers:

1. Which quality check showed the invented name most clearly?
2. Was the provided context wrong, or only the generated answer?
3. Are four test cases enough to approve a real application? Why not?

In [ ]:
# TODO: copy quality_rows, repair Q-03, and create a second run.
# fixed_rows = ...
# fixed_run = openai_client.evals.runs.create(...)


## Optional: explore further

Finished early? Add another quality check for whether the answer includes all required information, or replace the prepared context with results from Foundry IQ.

This is not required to complete the lab. In a real project, always record which version of the search system and test set produced the results.

## Cleanup (optional)

This cell can delete the evaluation and its runs.

Cleanup is off by default so you can still open and compare the reports. Run it only after saving the report links and completing your notes.

To remove the evaluation, set `WORKSHOP_ALLOW_CLEANUP=true` and run the cell. The code checks your namespace before deleting anything.

**You should see:** either a message that cleanup is disabled or a message confirming the deletion.

In [ ]:
allow_cleanup = os.getenv('WORKSHOP_ALLOW_CLEANUP', 'false').lower() == 'true'
if allow_cleanup:
    if not eval_name.endswith(f'-{resource_namespace}'):
        raise RuntimeError(f'Refusing to delete non-owned evaluation: {eval_name}')
    openai_client.evals.delete(eval_id=eval_object.id)
    print('Deleted this namespaced evaluation and its runs.')
else:
    print('Cleanup disabled so reports remain available for comparison.')